## Querying KG Project 6.21

To evaluate the KG, we formulated 10 competency questions (CQ). We preserved the initial query results in the commented-out cells below the SPARQL queries.
To demonstrate the analytical value of the KG, we formulated 4 queries.

In [ ]:
from rdflib import Graph

In [ ]:
path_to_kg = ""
kg = Graph()
kg.parse(path_to_kg, format="turtle")

In [ ]:
# KG stats
print(f"Total triples: {len(kg)}")
print(f"Subjects: {len(set(kg.subjects()))}")
print(f"Predicates: {len(set(kg.predicates()))}")
print(f"Objects: {len(set(kg.objects()))}")

In [ ]:
# Total triples: 13067
# Subjects: 2158
# Predicates: 46
# Objects: 6104

## Competency questions
1. How many monitoring records exist in total, and what is their distribution across years?
2. For each region, how many cases were recorded? Which regions have the most incidents?
3. What types of defendants (individual, official, legal_entity) are present, and how many cases involve each type?
4. How many records are linked to defendants, and how many lack defendant information?
5. Which admin articles were linked to records the most, and how are they distributed across years?
6. What types of penalties were imposed (fine, detention, warning, expulsion), and what is the distribution by years?
7. How many cases involved movies and TV series as trigger objects?
8. How many records have different source types?
9. Are there any monitoring records without such essential data as article, year, region?
10. How many defendants appear in more than one record?

In [ ]:
# Q1: How many monitoring records exist in total, and what is their distribution across years?
query_1 = """
PREFIX dataout: <https://dataout.org/ontology#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?year (COUNT(?record) AS ?numRecords)
WHERE {
  ?record a dataout:MonitoringRecord ;
          dataout:monitoredIncidentYear ?year .
}
GROUP BY ?year
ORDER BY ?year
"""

results = kg.query(query_1)
total = 0
for row in results:
    year = str(row.year).split('^')[0]
    count = int(row.numRecords)
    print(f"{year}: {count}")
    total += count
print(f"Total: {total}")

In [ ]:
# 2014: 6
# 2015: 4
# 2016: 4
# 2017: 2
# 2018: 1
# 2019: 3
# 2020: 3
# 2021: 3
# 2022: 4
# 2023: 99
# 2024: 70
# 2025: 117
# Total: 316

In [ ]:
# Q2: For each region, how many cases were recorded? Which regions have the most incidents?
query_2 = """
PREFIX dataout: <https://dataout.org/ontology#>
PREFIX schema: <https://schema.org/>
PREFIX region: <https://dataout.org/rdf/reference/ru_regions#>

SELECT ?regionName (COUNT(?record) AS ?numCases)
WHERE {
?record a dataout:MonitoringRecord ;
          schema:addressRegion ?regionUri .

?regionUri schema:name ?regionName .

FILTER (LANG(?regionName) = 'en')
}
GROUP BY ?regionName
ORDER BY DESC(?numCases)
"""

results = kg.query(query_2)
for row in results:
    region = row.regionName
    count = int(row.numCases)
    print(f"{region}: {count}")

In [ ]:
# Moscow: 193
# Saint-Petersburg: 27
# Krasnodar krai: 27
# Tatarstan Republic: 8
# Yaroslavl oblast: 6
# Khabarovsk krai: 3
# Sverdlovsk oblast: 3
# Novosibirsk oblast: 2
# Zabaykalsky krai: 2
# Volgograd oblast: 2
# Primorsky krai: 2
# Smolensk oblast: 2
# Yamalo-Nenets autonomous okrug: 2
# Nizhni Novgorod oblast: 2
# Omsk oblast: 2
# Kemerovo oblast: 2
# Irkutsk oblast: 2
# Penza oblast: 2
# Samara oblast: 2
# Tula oblast: 2
# Rostov oblast: 1
# Orenburg oblast: 1
# Mordovia Republic: 1
# Khanty-Mansijsk autonomous okrug: 1
# Kursk oblast: 1
# Kamchatka krai: 1
# Marij El Republic: 1
# Tambov oblast: 1
# Krasnoyarsk krai: 1
# Perm oblast: 1
# Komi Republic: 1
# Astrakhan oblast: 1
# Tyumen oblast: 1
# Murmansk oblast: 1
# Kaliningrad oblast: 1

In [ ]:
# Q3: What types of defendants (individual, official, legal_entity) are present, and how many cases involve each type?
query_3 = """
PREFIX dataout: <https://dataout.org/ontology#>

SELECT ?category (COUNT(DISTINCT ?defendant) AS ?numDefendants)
WHERE {
  ?defendant a dataout:Defendant ;
             dataout:defendantCategory ?category .
}
GROUP BY ?category
ORDER BY DESC(?numDefendants)
"""

results = kg.query(query_3)
for row in results:
    defendant_type = str(row.category).split("#")[-1]
    count = int(row.numDefendants)
    print(f"{defendant_type}: {count}")

In [ ]:
# individual: 115
# legalEntity: 52
# official: 41

In [ ]:
# Q4: How many records are linked to defendants, and how many lack defendant information?
query_4 = """
PREFIX dataout: <https://dataout.org/ontology#>

SELECT (COUNT(DISTINCT ?withDefendant) AS ?withDefendantCount) 
       (COUNT(DISTINCT ?withoutDefendant) AS ?withoutDefendantCount)
WHERE {
  {
    SELECT DISTINCT ?withDefendant
    WHERE {
      ?withDefendant a dataout:MonitoringRecord ;
                     dataout:hasDefendant ?defendant .
    }
  }
  UNION
  {
    SELECT DISTINCT ?withoutDefendant
    WHERE {
      ?withoutDefendant a dataout:MonitoringRecord .
      FILTER NOT EXISTS { ?withoutDefendant dataout:hasDefendant ?defendant . }
    }
  }
}
"""

results = kg.query(query_4)
for row in results:
    print(f"Records WITH defendants: {row.withDefendantCount}")
    print(f"Records WITHOUT defendants: {row.withoutDefendantCount}")

In [ ]:
# Records WITH defendants: 316
# Records WITHOUT defendants: 0

In [ ]:
# Q5: Which admin articles were linked to records the most, and how are they distributed across years?
query_5 = """
PREFIX dataout: <https://dataout.org/ontology#>
PREFIX law: <https://dataout.org/rdf/reference/ru_law#>

SELECT ?year ?articleUri (COUNT(?record) AS ?numRecords)
WHERE {
  ?record a dataout:MonitoringRecord ;
          dataout:concernsLegalArticle ?articleUri ;
          dataout:monitoredIncidentYear ?year .
}
GROUP BY ?year ?articleUri
ORDER BY ?year DESC(?numRecords)
"""

results = kg.query(query_5)
for row in results:
    year = str(row.year).split('^')[0]
    article = str(row.articleUri).split("#")[-1]
    count = int(row.numRecords)
    print(f"{year} - {article}: {count}")

In [ ]:
# 2014 - article_6_21_part_1: 4
# 2014 - article_6_21_part_2: 2
# 2015 - article_6_21_part_2: 4
# 2016 - article_6_21_part_2: 3
# 2016 - article_6_21_part_4: 1
# 2017 - article_6_21_part_2: 2
# 2018 - article_6_21_part_2: 1
# 2019 - article_6_21_part_1: 2
# 2019 - article_6_21_part_2: 1
# 2020 - article_6_21_part_2: 3
# 2021 - article_6_21_part_2: 3
# 2022 - article_6_21_part_2: 3
# 2022 - article_6_21_part_1: 1
# 2023 - article_6_21_2_part_2: 59
# 2023 - article_6_21_part_3: 24
# 2023 - article_6_21_part_7: 5
# 2023 - article_6_21_part_1: 5
# 2023 - article_6_21_part_8: 2
# 2023 - article_6_21_part_4: 2
# 2023 - article_6_21_2_part_4: 1
# 2023 - article_6_21_part_2: 1
# 2024 - article_6_21_part_3: 31
# 2024 - article_6_21_2_part_2: 18
# 2024 - article_6_21_part_1: 10
# 2024 - article_6_21_part_4: 3
# 2024 - article_6_21_part_7: 3
# 2024 - article_6_21_part_2: 3
# 2024 - article_6_21_part_5: 2
# 2025 - article_6_21_part_3: 74
# 2025 - article_6_21_part_4: 17
# 2025 - article_6_21_part_1: 15
# 2025 - article_6_21_part_7: 6
# 2025 - article_6_21_2_part_2: 5

In [ ]:
# Q6: What types of penalties were imposed, and what is the distribution by years?
query_6 = """
PREFIX dataout: <https://dataout.org/ontology#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?year ?penaltyTypeUri (COUNT(?penalty) AS ?numPenalties)
WHERE {
  ?record a dataout:MonitoringRecord ;
          dataout:monitoredIncidentYear ?year ;
          dataout:hasPenalty ?penalty .
  ?penalty dataout:penaltyCategory ?penaltyTypeUri .
}
GROUP BY ?year ?penaltyTypeUri
ORDER BY ?year ?numPenalties
"""

results = kg.query(query_6)
for row in results:
    year = str(row.year).split('^')[0]
    penalty_type = str(row.penaltyTypeUri).split("#")[-1]
    count = int(row.numPenalties)
    print(f"{year} - {penalty_type}: {count}")

In [ ]:
# 2014 - fine: 6
# 2015 - fine: 4
# 2016 - expulsion: 1
# 2016 - fine: 4
# 2017 - fine: 2
# 2018 - fine: 1
# 2019 - fine: 3
# 2020 - fine: 3
# 2021 - fine: 3
# 2022 - fine: 4
# 2023 - warning: 1
# 2023 - detention: 4
# 2023 - expulsion: 7
# 2023 - fine: 94
# 2024 - detention: 4
# 2024 - expulsion: 4
# 2024 - fine: 67
# 2025 - warning: 1
# 2025 - detention: 3
# 2025 - expulsion: 5
# 2025 - fine: 113

In [ ]:
# Q7: How many film and TV-series trigger objects are present?
query_7 = """
PREFIX dataout: <https://dataout.org/ontology#>

SELECT ?category (COUNT(DISTINCT ?triggerObject) AS ?totalTriggerObjects)
WHERE {
  ?triggerObject a dataout:TriggerObject ;
                 dataout:triggerObjectCategory ?category .
  FILTER(?category = dataout:film || ?category = dataout:tvSeries)
}
GROUP BY ?category
ORDER BY ?category
"""

results = kg.query(query_7)
for row in results:
    category = str(row.category).split("#")[-1]
    count = int(row.totalTriggerObjects)
    print(f"{category}: {count}")

In [ ]:
# film: 57
# tvSeries: 10

In [ ]:
# Q8: How many records have different source types?
query_8 = """
PREFIX dataout: <https://dataout.org/ontology#>
PREFIX prov: <http://www.w3.org/ns/prov#>

SELECT ?category (COUNT(DISTINCT ?record) AS ?numRecords)
WHERE {
  ?record a dataout:MonitoringRecord ;
          prov:wasDerivedFrom ?source .
  ?source dataout:sourceCategory ?category .
}
GROUP BY ?category
ORDER BY DESC(?numRecords)
"""

results = kg.query(query_8)
for row in results:
    source_type = str(row.category).split("#")[-1]
    count = int(row.numRecords)
    print(f"{source_type}: {count}")

In [ ]:
# courtDocument: 316
# massMediaPublication: 53
# socialMediaPost: 35
# pressRelease: 2

In [ ]:
# Q9: Are there any monitoring records without essential data (article, year, region)?
query_9 = """
PREFIX dataout: <https://dataout.org/ontology#>
PREFIX schema: <https://schema.org/>

SELECT (COUNT(DISTINCT ?noArticle) AS ?missingArticles)
       (COUNT(DISTINCT ?noYear) AS ?missingYears)
       (COUNT(DISTINCT ?noRegion) AS ?missingRegions)
WHERE {
  {
    SELECT DISTINCT ?noArticle
    WHERE {
      ?noArticle a dataout:MonitoringRecord .
      FILTER NOT EXISTS { ?noArticle dataout:concernsLegalArticle ?article . }
    }
  }
  UNION
  {
    SELECT DISTINCT ?noYear
    WHERE {
      ?noYear a dataout:MonitoringRecord .
      FILTER NOT EXISTS { ?noYear dataout:monitoredIncidentYear ?year . }
    }
  }
  UNION
  {
    SELECT DISTINCT ?noRegion
    WHERE {
      ?noRegion a dataout:MonitoringRecord .
      FILTER NOT EXISTS { ?noRegion schema:addressRegion ?region . }
    }
  }
}
"""

results = kg.query(query_9)
for row in results:
    print(f"No article: {row.missingArticles}")
    print(f"No year: {row.missingYears}")
    print(f"No region: {row.missingRegions}")

In [ ]:
# No article: 0
# No year: 0
# No region: 0

In [ ]:
# Q10: How many defendants appear in more than one record?
query_10 = """
PREFIX dataout: <https://dataout.org/ontology#>

SELECT ?defendant (COUNT(?record) AS ?recordCount)
WHERE {
  ?record a dataout:MonitoringRecord ;
          dataout:hasDefendant ?defendant .
}
GROUP BY ?defendant
HAVING (COUNT(?record) > 1)
"""

results = kg.query(query_10)
print(len(results))

In [ ]:
# 39

## Analytical queries

In [ ]:
# add the ontology
path_to_ontology = "/dataout-org/ontology/ontology.ttl"
kg.parse(path_to_ontology, format="turtle")

### Q2.1: Group cases by defendant category and trigger object category: In how many cases defendants of a particular category were prosecuted for each category of trigger objects?

In [ ]:
q_2_1 = """

PREFIX dataout: <https://dataout.org/ontology#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?defendant_cat_label ?trigger_obj_cat_label
       (COUNT(DISTINCT ?record) AS ?numRecords)

WHERE {
    ?record a dataout:MonitoringRecord ;
            dataout:hasDefendant ?defendant ;
            dataout:hasTriggerObject ?trigger_object .

    ?defendant dataout:defendantCategory / skos:prefLabel ?defendant_cat_label .
    ?trigger_object dataout:triggerObjectCategory / skos:prefLabel ?trigger_obj_cat_label .
}

GROUP BY ?defendant_cat_label ?trigger_obj_cat_label
ORDER BY ?defendant_cat_label DESC(?numRecords)

"""

results = kg.query(q_2_1)
for row in results:
    print(row.defendant_cat_label, row.trigger_obj_cat_label, row.numRecords)

In [ ]:
# Individual Social media post 122
# Individual Public action 16
# Individual Dating profile 10
# Individual Queer venue 6
# Individual Web content 5
# Individual Protest 3
# Individual Merchandise 1
# Legal entity Film 39
# Legal entity Comics 15
# Legal entity TV series 11
# Legal entity Music video 10
# Legal entity Book 8
# Legal entity Queer venue 4
# Legal entity TV programme 3
# Legal entity Merchandise 3
# Legal entity Social media post 2
# Legal entity Mass media publication 2
# Legal entity Web content 1
# Official Film 23
# Official Comics 13
# Official Mass media publication 8
# Official Music video 7
# Official TV series 4
# Official Queer venue 3
# Official Social media post 3
# Official Web content 2
# Official Merchandise 1
# Official TV programme 1
# Official Book 1

### Q2.2: Same trigger objects across cases

In [ ]:
q_2_2 = """
PREFIX dataout: <https://dataout.org/ontology#>
PREFIX schema: <https://schema.org/>

SELECT ?trigger_obj_id (COUNT(DISTINCT ?record) AS ?n_records)

WHERE {

    ?trigger_obj a dataout:TriggerObject ;
	schema:identifier ?trigger_obj_id .

	?record a dataout:MonitoringRecord ;
	dataout:hasTriggerObject ?trigger_obj .
}

GROUP BY ?trigger_obj_id
HAVING (COUNT(?record) > 1)
ORDER BY DESC(?n_records)
"""

results_q_2_2 = kg.query(q_2_2)
print(f"N trigger objects appearing in more than one case: {len(results_q_2_2)}")

for row in results_q_2_2:
    print(row.trigger_obj_id, row.n_records)

In [ ]:
# N trigger objects appearing in more than one case: 63
# 17203 14
# 381341 10
# 680590 7
# 201765 6
# 523773 6
# 9801 6
# 10763 4
# 127008 4
# 37735 4
# 554100 4
# 9785444817810 4
# OL35877617M 4
# 167032 3
# 1966 3
# 245775 3
# 246860 3
# 567598 3
# 64122 3
# 666622 3
# OL20004466W 3
# OL22355901W 3
# OL24826887M 3
# comics_002 3
# 130273 2
# 273169 2
# 31841 2
# 338189 2
# 43593 2
# 487498 2
# 490132 2
# 517148 2
# 55347 2
# 586707 2
# 588182 2
# 612294 2
# 711577 2
# 752 2
# 795811 2
# 80840 2
# 901908 2
# 94115 2
# ExJmET8boVw 2
# NcfFV9eesZI 2
# comics_003 2
# comics_004 2
# comics_005 2
# comics_006 2
# comics_007 2
# comics_008 2
# comics_009 2
# comics_010 2
# comics_011 2
# comics_013 2
# comics_014 2
# film_001 2
# mLJM9jLwo2c 2
# mass_media_008 2
# merch_005 2
# public_action_003 2
# public_action_004 2
# social_media_037 2
# tv_programme_002 2
# vJIJdAgbGCU 2

### Q2.3: Prosecution against legal entities and affiliated employees for the same trigger objects

The number of instances when legal entities and affiliated individuals share the same trigger objects (in cases when there are several trigger objects, they may not overlap fully).

In [ ]:
q_2_3 = """
PREFIX dataout: <https://dataout.org/ontology#>
PREFIX schema: <https://schema.org/>

SELECT ?record_official_id ?official_id ?record_legal_id ?legal_entity_id (COUNT(DISTINCT ?trigger_object) AS ?n_shared_triggers)

WHERE {

	{
		SELECT DISTINCT
		?record_official_id ?official_id ?legal_entity ?trigger_object
		
		WHERE {
			?record_official a dataout:MonitoringRecord ;
			dataout:monitoringRecordNumber ?record_official_id ;
			dataout:hasDefendant ?official ;
			dataout:hasTriggerObject ?trigger_object .

			?official dataout:defendantCategory dataout:official ;
			schema:affiliation ?legal_entity ;
			schema:identifier ?official_id .
		}

	}

	{
		SELECT DISTINCT
		?record_legal_id ?legal_entity_id ?legal_entity ?trigger_object

		WHERE {
				?record_legal a dataout:MonitoringRecord ;
				dataout:monitoringRecordNumber ?record_legal_id ;
				dataout:hasDefendant ?legal_entity ;
				dataout:hasTriggerObject ?trigger_object .

				?legal_entity dataout:defendantCategory dataout:legalEntity ;
				schema:identifier ?legal_entity_id .
		}
	}

}

GROUP BY ?record_official_id ?record_legal_id ?official_id ?legal_entity_id
"""

results_q_2_3 = kg.query(q_2_3)
k = 0
for row in results_q_2_3:
    k += 1
    print(f"{k}. LE: Case {row.record_legal_id}, Def. {row.legal_entity_id} | Official: Case {row.record_official_id}, Def. {row.official_id} | N Shared Triggers: {row.n_shared_triggers}")

In [ ]:
# 1. LE: Case 5-190/2024, Def. 1187746527099 | Official: Case 5-1189/23, Def. official_0008 | N Shared Triggers: 1
# 2. LE: Case 5-893/23, Def. 1027700198767 | Official: Case 05-4573/2023, Def. official_0013 | N Shared Triggers: 1
# 3. LE: Case 5-1242/2023, Def. 1057747513680 | Official: Case 5-840/23, Def. official_0015 | N Shared Triggers: 1
# 4. LE: Case 5-493/23, Def. 1057747513680 | Official: Case 5-553/23, Def. official_0016 | N Shared Triggers: 1
# 5. LE: Case 05-6777/2023, Def. 1067746687623 | Official: Case 5-1073/23, Def. official_0017 | N Shared Triggers: 1
# 6. LE: Case 05-6398/2023, Def. 1077757496937 | Official: Case 5-955/23, Def. official_0018 | N Shared Triggers: 1
# 7. LE: Case 05-1608/2023, Def. 1077759854919 | Official: Case 5-957/23, Def. official_0020 | N Shared Triggers: 2
# 8. LE: Case 05-0516/2025, Def. 1077759854919 | Official: Case 5-647/2025, Def. official_0022 | N Shared Triggers: 1
# 9. LE: Case 05-0835/2023, Def. 1087746540705 | Official: Case 5-999/23, Def. official_0023 | N Shared Triggers: 1
# 10. LE: Case 05-0268/2025, Def. 1087746888833 | Official: Case 05-0269/2025, Def. official_0024 | N Shared Triggers: 6
# 11. LE: Case 05-1225/2023, Def. 1097746597046 | Official: Case 5-628/23, Def. official_0025 | N Shared Triggers: 1
# 12. LE: Case 5-302/2024, Def. 1127847522110 | Official: Case 12-1609/2023, Def. official_0026 | N Shared Triggers: 1
# 13. LE: Case 05-0583/2023, Def. 1187746527099 | Official: Case 5-940/23, Def. official_0028 | N Shared Triggers: 2
# 14. LE: Case 05-1040/2023, Def. 1197746708917 | Official: Case 5-641/23, Def. official_0029 | N Shared Triggers: 2
# 15. LE: Case 05-0486/2023, Def. 1237700009700 | Official: Case 5-881/23, Def. official_0030 | N Shared Triggers: 3
# 16. LE: Case 7-149/2014, Def. 1026700647885 | Official: Case 12-28/2014, Def. official_0038 | N Shared Triggers: 1
# 17. LE: Case 05-0067/2025, Def. 1027739048281 | Official: Case 05-1326/374/2024, Def. official_0040 | N Shared Triggers: 1
# 18. LE: Case 05-0915/2023, Def. 1027809169585 | Official: Case 5-633/23, Def. official_0014 | N Shared Triggers: 1
# 19. LE: Case 05-1607/2023, Def. 1077759854919 | Official: Case 5-1020/23, Def. official_0021 | N Shared Triggers: 4
# 20. LE: Case 05-1459/2023, Def. 1077759854919 | Official: Case 5-936/23, Def. official_0021 | N Shared Triggers: 10
# 21. LE: Case 05-0771/2023, Def. 1077758948112 | Official: Case 5-1406/23, Def. official_0019 | N Shared Triggers: 1
# 22. LE: Case 05-1147/2023, Def. 1077758948112 | Official: Case 5-954/23, Def. official_0019 | N Shared Triggers: 2
# 23. LE: Case 05-0902/2023, Def. 1077758948112 | Official: Case 5-1479/23, Def. official_0019 | N Shared Triggers: 4
# 24. LE: Case 5-253/2024, Def. 1167847381130 | Official: Case 5-1423/2024-162, Def. official_0027 | N Shared Triggers: 3
# 25. LE: Case 5-849/2024, Def. 1167847381130 | Official: Case 5-837/2024-162, Def. official_0027 | N Shared Triggers: 1
# 26. LE: Case 5-2518/2025-162, Def. 1167847381130 | Official: Case 5-2433/2025-162, Def. official_0027 | N Shared Triggers: 1
# 27. LE: Case 5-1661/2025-162, Def. 1167847381130 | Official: Case 5-1663/2025-162, Def. official_0027 | N Shared Triggers: 1
# 28. LE: Case 5-348/2025, Def. 1167847381130 | Official: Case 5-683/2025-162, Def. official_0027 | N Shared Triggers: 1
# 29. LE: Case 05-0658/2025, Def. 1237700761539 | Official: Case 05-0755/374/2025, Def. official_0031 | N Shared Triggers: 1
# 30. LE: Case 05-0656/2025, Def. 1237700761539 | Official: Case 05-0751/374/2025, Def. official_0031 | N Shared Triggers: 1
# 31. LE: Case 05-0600/2025, Def. 1237700761539 | Official: Case 05-0675/374/2025, Def. official_0031 | N Shared Triggers: 1
# 32. LE: Case 05-0655/2025, Def. 1237700761539 | Official: Case 05-0750/374/2025, Def. official_0031 | N Shared Triggers: 1
# 33. LE: Case 05-0605/2025, Def. 1237700761539 | Official: Case 05-0674/374/2025, Def. official_0031 | N Shared Triggers: 1
# 34. LE: Case 05-0606/2025, Def. 1237700761539 | Official: Case 05-0681/374/2025, Def. official_0031 | N Shared Triggers: 1
# 35. LE: Case 05-0661/2025, Def. 1237700761539 | Official: Case 05-0752/374/2025, Def. official_0031 | N Shared Triggers: 1
# 36. LE: Case 05-0487/2025, Def. 1237700761539 | Official: Case 05-0611/374/2025, Def. official_0031 | N Shared Triggers: 1
# 37. LE: Case 05-0659/2025, Def. 1237700761539 | Official: Case 05-0754/374/2025, Def. official_0031 | N Shared Triggers: 1
# 38. LE: Case 05-0675/2025, Def. 1237700761539 | Official: Case 05-0794/374/2025, Def. official_0031 | N Shared Triggers: 1
# 39. LE: Case 05-0651/2025, Def. 1237700761539 | Official: Case 05-0749/374/2025, Def. official_0031 | N Shared Triggers: 1

### Q2.4: Defendants-individuals with more than one case

In [ ]:
q_2_4 = """
PREFIX dataout: <https://dataout.org/ontology#>
PREFIX schema: <https://schema.org/>

SELECT ?defendant_id (COUNT(DISTINCT ?record) AS ?num_records)

WHERE {

	?defendant a dataout:Defendant ;
    dataout:defendantCategory dataout:individual ;
	schema:identifier ?defendant_id .

	?record a dataout:MonitoringRecord ;
	dataout:hasDefendant ?defendant .

}

GROUP BY ?defendant_id
HAVING (COUNT(DISTINCT ?record) > 1)
ORDER BY DESC(?num_records)
"""

results_q_2_4 = kg.query(q_2_4)
print(f"N defendants-individuals appearing in more than one case: {len(results_q_2_4)}")

total_court_cases = 0
for row in results_q_2_4:
    total_court_cases += int(row.num_records)
    print(row.defendant_id, row.num_records)
print(f"Total court cases: {total_court_cases}")

In [ ]:
# N defendants-individuals appearing in more than one case: 16
# individual_0026 9
# individual_0052 7
# individual_0104 6
# individual_0077 5
# individual_0059 4
# individual_0065 4
# individual_0044 3
# individual_0064 3
# individual_0097 3
# individual_0098 3
# individual_0024 2
# individual_0048 2
# individual_0053 2
# individual_0076 2
# individual_0095 2
# individual_0100 2
# Total court cases: 59